# Outstanding Questions

## Currency

Currently, I only select USD charges. How should I treat CAD? Add as-is, or convert to USD at what rate?
* https://pypi.org/project/CurrencyConverter/

## Thresholds

Currently, tier thresholds are static, copied from Tim's worksheet, but they are supposed to represent percentages of the total. Should I dynamically create thresholds based on total net revenue?
* Bronze (0.1% of book)
* Silver (0.25% of book)
* Gold (1% of book)
* Diamond (4% of book)

## Refunds

Currently, due to the state of refunds, I'm including charges with status REFUND_DOWNGRADE_PENDING and REFUND_DOWNGRADE_IN_PROGRESS along with REFUNDED_DOWNGRADE. Should I apply the same "it only counts if we actually debit the money" approach?

## Semi-Integrators/Enterprise Developers

How should semi-integrators/enterprise devs be indentified and removed from this analysis?

In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py

# Install a pip package in the current Jupyter kernel.
import sys
!{sys.executable} -m pip install --user pypika

In [ ]:
import datetime
from dateutil.relativedelta import relativedelta
from pypika import MySQLQuery, Table, Order, Case, functions as fn

def create_query_appmkt_3p_summed_charge_amount(charge_statuses, charge_currency, amount_as, start_date, end_date):
    charge = Table("charge")
    merchant_app_charge = Table("merchant_app_charge")
    merchant_app = Table("merchant_app")
    developer_app = Table("developer_app")
    developer = Table("developer")

    total_amount = (fn.Sum(charge.amount)/100).as_(amount_as)
    developer_portion = (fn.Sum(fn.Coalesce(charge.developer_portion,
                                              charge.amount * 0.7))/100).as_("developer_portion")
    clover_portion = (fn.Sum(charge.amount - fn.Coalesce(charge.developer_portion,
                                                           charge.amount * 0.7))/100).as_("clover_portion")
    
    q = MySQLQuery.from_(charge).join(
        merchant_app_charge
    ).on(
        merchant_app_charge.charge_id == charge.id 
    ).join(
        merchant_app
    ).on(
        merchant_app.id == merchant_app_charge.merchant_app_id
    ).join(
        developer_app
    ).on(
        developer_app.id == merchant_app.app_id
    ).join(
        developer
    ).on(
        developer.id == developer_app.developer_id
    ).select(
        developer.id,
        developer.uuid,
        developer.name,
        charge.currency,
        total_amount
    ).where(
        (developer.name != "Clover") &
        (developer.is_rev_share_flat_rate == 0)
    ).where(
        (charge.system_type == "INFOLEASE") &
        (charge.status.isin(charge_statuses)) &
        (charge.currency == charge_currency)
    ).where(
        # charge.created_time is indexed.
        (charge.created_time >= fn.Date(start_date)) &
        (charge.created_time < fn.Date(end_date))
    ).groupby(developer.id) \
    .orderby(total_amount, order=Order.desc)

    print(q)
    query = q.get_sql()
    return query

# Year to date
start_date = datetime.datetime.utcnow().replace(month=1, day=1).date()
end_date = start_date + relativedelta(years=1)

# Only USD for now
charge_currency = "USD"

# Query for gross rev
amount_as = "gross_revenue"
charge_statuses = ["DISBURSED", "BILLED"]
gross_query = create_query_appmkt_3p_summed_charge_amount(charge_statuses=charge_statuses,
                                                          charge_currency=charge_currency,
                                                          amount_as=amount_as,
                                                          start_date=start_date,
                                                          end_date=end_date)
# Query for total refunds
amount_as = "total_refunds"
charge_statuses = ["REFUND_DOWNGRADE_PENDING", "REFUND_DOWNGRADE_IN_PROGRESS", "REFUNDED_DOWNGRADE"]
refund_query = create_query_appmkt_3p_summed_charge_amount(charge_statuses=charge_statuses,
                                                           charge_currency=charge_currency,
                                                           amount_as=amount_as,
                                                           start_date=start_date,
                                                           end_date=end_date)

db = Db("~/.clover/p801.cfg")
df = pd.read_sql(gross_query, con=db.conn)
refunds = pd.read_sql(refund_query, con=db.conn)
db.close()

df = df.set_index("id").join(refunds.filter(["id", "total_refunds"]).set_index("id"))
df.head()

In [ ]:
def create_query_appmkt_3p_net_installs():  # TODO: Add start and end date optional params?
    merchant_app = Table("merchant_app")
    merchant = Table("merchant")
    merchant_gateway = Table("merchant_gateway")
    payment_processor = Table("payment_processor")
    developer_app = Table("developer_app")
    developer = Table("developer")

    net_installs = fn.Sum(
        Case()
        .when(merchant_app.deleted_time.isnull(), 1)
        .else_(0)
    ).as_("net_installs")

    q = MySQLQuery.from_(merchant_app).join(
        merchant
    ).on(
        merchant.id == merchant_app.merchant_id
    ).join(
        merchant_gateway
    ).on(
        merchant_gateway.merchant_id == merchant_app.merchant_id
    ).join(
        payment_processor
    ).on(
        payment_processor.id == merchant_gateway.payment_processor_id
    ).join(
        developer_app
    ).on(
        developer_app.id == merchant_app.app_id
    ).join(
        developer
    ).on(
        developer.id == developer_app.developer_id  
    ).select(
        developer.id,
        developer.uuid,
        developer.name,
        net_installs
    ).where(
        (developer.name != "Clover") &
        (developer.is_rev_share_flat_rate == 0) &
        (developer.approval_status == "APPROVED")
    ).where(
        (payment_processor.production == 1) &
        (merchant.infolease_suppress_billing != 1)    
    ).groupby(developer.id) \
    .orderby(net_installs, order=Order.desc)

    print(q)
    query = q.get_sql()
    return query

installs_query = create_query_appmkt_3p_net_installs()
db = Db("~/.clover/p801.cfg")
installs = pd.read_sql(installs_query, con=db.conn)
db.close()

installs.drop(installs[installs.net_installs < 2].index, inplace=True)
df = df.merge(installs.set_index("id"), how="outer")
df.sort_values(by="net_installs", ascending=False, inplace=True)
df.head()

In [ ]:
# Fill NaN and calculate net revenue.
df.fillna({'currency': charge_currency}, inplace=True)  # Filling for rows introduced by adding installs.
df.fillna(0, inplace=True)
df["net_revenue"] = df.apply(lambda row: row.gross_revenue - row.total_refunds, axis="columns")
df.drop(["gross_revenue", "total_refunds"], axis="columns", inplace=True)
df.head()

In [ ]:
semi_integrator_uuids = ["04A8QFQ4M0G9E",  # All My Sons
                         "RXSDNAM1F5B3R",  # Amazon Pay
                         "69EH6FMAY6MY0",  # Booker
                         "R5C8Q9XE2JD5R",  # Bypass Mobile
                         "ZMDHGNC9154JC",  # CareCloud
                         "KNB4ZK9BQSJYG",  # Clubspeed
                         "X9G2XSTRRFZW6",  # Daxko
                         "3WY453FAESHG8",  # Ideal
                         "A8DZ831KSN992",  # PetExec
                         "DCZZKW9GASKBC",  # Prosperity
                         "0WZ9YVB55QXFT",  # SAP Anywhere
                         "2DQA9HHFN0Y70",  # ShopKeep
                         "MGNQ0SNWV4QBG",  # Solupay
                         "RHGVEGPYT3FQ4",  # Springboard Retail
                         "3PZQEJNSB3Y5M"]  # UpdatePromise

# TODO: Uncomment to remove known semi-integrators/enterprise developers.
# df = df[~df["uuid"].isin(semi_integrator_uuids)]

In [ ]:
# Once all joins are done, combine developer accounts into developer organizations.
# After this step, you cannot join any more data frames on developer id (or uuid).
aggregation_functions = {"uuid": lambda x: ", ".join(x),
                         "name": "first",
                         "currency": "first",
                         "net_revenue": "sum",
                         "net_installs": "sum"}
df = df.groupby(df["name"]).aggregate(aggregation_functions).reset_index(drop=True)
df = df[["uuid", "name", "currency", "net_revenue", "net_installs"]]
df.head()

In [ ]:
import datetime
import calendar

def annualize_ytd(value):
    current = datetime.datetime.utcnow()
    day_of_year = current.timetuple().tm_yday
    total_days = 366 if calendar.isleap(current.year) else 365
    year_progress = float(day_of_year)/float(total_days)
    return value/year_progress

df["annualized_net_revenue"] = df["net_revenue"].map(lambda x: round(annualize_ytd(x)))
df = df.sort_values(by="annualized_net_revenue", ascending=False).reset_index(drop=True)
df.head()

In [ ]:
tier_to_int = {"Diamond": 4, "Gold": 3, "Silver": 2, "Bronze": 1, "Public": 0}
int_to_tier = {v: k for k, v in tier_to_int.items()}

DIAMOND_REV_THRESHOLD = 2400000
GOLD_REV_THRESHOLD    =  600000
SILVER_REV_THRESHOLD  =  250000
BRONZE_REV_THRESHOLD  =   50000

def categorize_by_rev(value):
    if value > DIAMOND_REV_THRESHOLD:
        return tier_to_int["Diamond"]
    if value > GOLD_REV_THRESHOLD:
        return tier_to_int["Gold"]
    if value > SILVER_REV_THRESHOLD:
        return tier_to_int["Silver"]
    if value > BRONZE_REV_THRESHOLD:
        return tier_to_int["Bronze"]
    else:
        return tier_to_int["Public"]
    
DIAMOND_INSTALLS_THRESHOLD = 50000
GOLD_INSTALLS_THRESHOLD    =  6000
SILVER_INSTALLS_THRESHOLD  =  1500
BRONZE_INSTALLS_THRESHOLD  =   300

def categorize_by_installs(value):
    if value > DIAMOND_INSTALLS_THRESHOLD:
        return tier_to_int["Diamond"]
    if value > GOLD_INSTALLS_THRESHOLD:
        return tier_to_int["Gold"]
    if value > SILVER_INSTALLS_THRESHOLD:
        return tier_to_int["Silver"]
    if value > BRONZE_INSTALLS_THRESHOLD:
        return tier_to_int["Bronze"]
    else:
        return tier_to_int["Public"]

df["support_tier_by_rev"] = df["annualized_net_revenue"].map(lambda x: categorize_by_rev(x))
df["support_tier_by_installs"] = df["net_installs"].map(lambda x: categorize_by_installs(x))
df.head()

In [ ]:
# TODO: Uncomment to drop developers who are in 'Public' tier by both criteria.
# df.drop(df[(df.support_tier_by_rev == 0) & (df.support_tier_by_installs == 0)].index, inplace=True)

df_sorted = df.groupby(["support_tier_by_rev"], sort=False, as_index=False).apply(lambda x: x.sort_values(["net_installs"], ascending=False)).reset_index(drop=True)
df_named_tiers = df_sorted.replace({"support_tier_by_rev":int_to_tier, "support_tier_by_installs":int_to_tier})
df_named_tiers

In [ ]:
output_df = df_named_tiers
output_loc = "/home/rachel/app_market_rankings.csv"
try:
    output_df.to_csv(output_loc)
except UnicodeEncodeError:
    output_df.to_csv(output_loc, encoding="utf-8")